In [36]:
import sqlite3
import pandas as pd

# Connect to database
conn = sqlite3.connect('sales.db')

# Export to CSV
df = pd.read_sql_query("SELECT * FROM sales", conn)
df.to_csv("sales.csv", index=False)

# Read CSV for analysis
data = pd.read_csv("sales.csv")

# Select subset (e.g., only 'North' region)
subset = data[data['region'] == 'North']
print(subset)


   id   product region        date  units_sold  revenue
0   1  Widget A  North  2025-01-01          10    100.0
2   3  Widget A  North  2025-01-02          12    120.0


In [ ]:
!pip install dash

  Using cached flask-3.1.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached werkzeug-3.1.4-py3-none-any.whl.metadata (4.0 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
   ---------------------------------------- 0.0/7.9 MB ? eta -:--:--
   ----- ---------------------------------- 1.0/7.9 MB 5.2 MB/s eta 0:00:02
   --------- ------------------------------ 1.8/7.9 MB 4.2 MB/s eta 0:00:02
   --------- ------------------------------ 1.8/7.9 MB 4.2 MB/s eta 0:00:02
   --------- ------------------------------ 1.8/7.9 MB 4.2 MB/s eta 0:00:02
   --------- ------------------------------ 1.8/7.9 MB 4.2 MB/s eta 0:00:02
   --------- ------------------------------ 1.8/7.9 MB 4.2 MB/s eta 0:00:02
   --------- ------------------------------ 1.8/7.9 MB 4.2 MB/s eta 0:00:02
   --------- ------------------------------ 1.8/7.9 MB 4.2 MB/s eta 0:00:02
   --------- ------------------------------ 1.8/7.9 MB 4.2 MB/s eta 0:00:02
   --------- ------------------------------ 1.8/7.9

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [ ]:
!pip install jupyter-dash


   ---------------------------------------- 0/2 [ansi2html]
   ---------------------------------------- 0/2 [ansi2html]
   ---------------------------------------- 0/2 [ansi2html]
   ---------------------------------------- 0/2 [ansi2html]
   ---------------------------------------- 0/2 [ansi2html]
   ---------------------------------------- 0/2 [ansi2html]
   ---------------------------------------- 0/2 [ansi2html]
   ---------------------------------------- 0/2 [ansi2html]
   ---------------------------------------- 0/2 [ansi2html]
   ---------------------------------------- 0/2 [ansi2html]
   ---------------------------------------- 0/2 [ansi2html]
   ---------------------------------------- 0/2 [ansi2html]
   -------------------- ------------------- 1/2 [jupyter-dash]
   -------------------- ------------------- 1/2 [jupyter-dash]
   ---------------------------------------- 2/2 [jupyter-dash]

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [31]:
from dash import Dash, dcc, html
import plotly.express as px

app = Dash(__name__)

fig = px.line(subset, x='date', y='units_sold', title='Units Sold Over Time')
fig2 = px.line(forecast, x='date', y='predicted_units_sold', title='Predicted Units Sold')

app.layout = html.Div([
    html.H1("Sales Dashboard"),
    dcc.Graph(figure=fig),
    dcc.Graph(figure=fig2)
])

app.run(debug=True)

In [ ]:
from openai import OpenAI
import os

# Set your API key (replace with your actual key)
# Option 1: Direct (less secure)
api_key = "YOUR_ACTUAL_API_KEY_HERE"

# Option 2: From environment variable (more secure)
# api_key = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=api_key)

# Convert historical data to prompt
history = subset[['date','units_sold']].to_dict(orient='records')
prompt = f"Predict the next 7 days of units sold based on historical sales data: {history}"

try:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role":"user", "content": prompt}]
    )
    genai_forecast = response.choices[0].message.content
    print(genai_forecast)
except Exception as e:
    print(f"Error: {e}")
    print("Make sure you've added a valid API key")


To predict the next 7 days of units sold based on the limited historical sales data provided, we can use a simple linear approach since we only have two data points. 

From the data:

- On 2025-01-01, units sold = 10
- On 2025-01-02, units sold = 12

We can see an increase of 2 units sold from January 1 to January 2. If we assume this trend of increasing sales continues, we could project daily sales using this incremental approach.

### Prediction:
- **2025-01-03**: 12 + 2 = 14
- **2025-01-04**: 14 + 2 = 16
- **2025-01-05**: 16 + 2 = 18
- **2025-01-06**: 18 + 2 = 20
- **2025-01-07**: 20 + 2 = 22
- **2025-01-08**: 22 + 2 = 24
- **2025-01-09**: 24 + 2 = 26

### Summary of Units Sold
- **2025-01-03**: 14 units
- **2025-01-04**: 16 units
- **2025-01-05**: 18 units
- **2025-01-06**: 20 units
- **2025-01-07**: 22 units
- **2025-01-08**: 24 units
- **2025-01-09**: 26 units

Please note that this is a simple linear extrapolation and may not reflect actual future sales accurately, especially wi

In [42]:
# Parse GenAI output and create forecast DataFrame
import re

future_dates = pd.date_range(start='2025-01-03', periods=7)

try:
    # Extract numbers from AI response text
    numbers = re.findall(r'\b\d+\b', genai_forecast)
    predicted_units = [int(n) for n in numbers[:7]]  # Get first 7 numbers
    
    # If we got fewer than 7 values, pad with reasonable defaults
    if len(predicted_units) < 7:
        avg_value = sum(predicted_units) / len(predicted_units) if predicted_units else 12
        predicted_units.extend([int(avg_value)] * (7 - len(predicted_units)))
    
    print(f"Parsed {len(predicted_units)} predictions from GenAI response")
    
except Exception as e:
    print(f"Could not parse GenAI response: {e}")
    print("Using fallback values...")
    predicted_units = [10, 12, 15, 13, 14, 16, 18]

forecast_genai = pd.DataFrame({
    'date': future_dates,
    'predicted_units_sold': predicted_units
})

print(forecast_genai)


Parsed 7 predictions from GenAI response
        date  predicted_units_sold
0 2025-01-03                     7
1 2025-01-04                  2025
2 2025-01-05                     1
3 2025-01-06                     1
4 2025-01-07                    10
5 2025-01-08                  2025
6 2025-01-09                     1


In [43]:
import plotly.graph_objects as go

# Create simple plotly figures (no Dash needed)
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=subset['date'], y=subset['units_sold'], mode='lines', name='Actual Sales'))
fig1.update_layout(title='Actual Sales Over Time', xaxis_title='Date', yaxis_title='Units Sold', height=400)

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=forecast_genai['date'], y=forecast_genai['predicted_units_sold'], mode='lines+markers', name='Predicted Sales'))
fig2.update_layout(title='Predicted Units Sold (Next 7 Days)', xaxis_title='Date', yaxis_title='Units Sold', height=400)

# Display the figures
print("=== Sales Dashboard ===\n")
print("Actual Sales:")
fig1.show()

print("\n\nPredicted Sales:")
fig2.show()

print("\n✓ Dashboard displayed successfully!")

=== Sales Dashboard ===

Actual Sales:




Predicted Sales:



✓ Dashboard displayed successfully!


In [19]:
from fastapi import FastAPI
import pandas as pd
import sqlite3
from sklearn.linear_model import LinearRegression
from datetime import timedelta

app = FastAPI(title="Sales Forecast API")

# -------------------------------
# Load Data from SQLite
# -------------------------------
def load_sales_data():
    conn = sqlite3.connect('sales.db')
    df = pd.read_sql_query("SELECT * FROM sales", conn)
    conn.close()
    return df

# -------------------------------
# Endpoint: Get Raw Sales Data
# -------------------------------
@app.get("/sales")
def get_sales():
    df = load_sales_data()
    return df.to_dict(orient='records')

# -------------------------------
# Endpoint: Predict Future Sales
# -------------------------------
@app.get("/predict")
def predict_sales(region: str = "North"):
    df = load_sales_data()
    subset = df[df['region'] == region]
    subset['date_ordinal'] = pd.to_datetime(subset['date']).map(pd.Timestamp.toordinal)

    X = subset[['date_ordinal']]
    y = subset['units_sold']
    model = LinearRegression()
    model.fit(X, y)

    future_dates = pd.date_range(start=pd.to_datetime(subset['date']).max() + timedelta(days=1), periods=7)
    future_X = future_dates.map(pd.Timestamp.toordinal).values.reshape(-1, 1)
    predictions = model.predict(future_X)

    # Simulate GenAI
    genai_predictions = [int(p + 2) for p in predictions]

    forecast_genai = pd.DataFrame({
        'date': future_dates,
        'predicted_units_sold': genai_predictions
    })
    return forecast_genai.to_dict(orient='records')

# -------------------------------
# To run the API server, use in terminal:
# uvicorn sales:app --reload --host 0.0.0.0 --port 8000
# 
# Then test the API in another cell with:
# import requests
# response = requests.get('http://localhost:8000/sales')
# print(response.json())
# 
# Or visit: http://localhost:8000/docs
# -------------------------------

In [22]:
!pip install requests

import requests
import time

# Wait for API to be ready
time.sleep(1)

try:
    # Get all sales data
    response = requests.get('http://localhost:8000/sales')
    sales_data = response.json()
    print("Sales Data:")
    print(sales_data[:2])  # Show first 2 records
except Exception as e:
    print(f"Error: {e}")
    print("Make sure FastAPI server is running first!")
    print("Run in terminal: uvicorn sales:app --reload")

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\wwwdh\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Sales Data:
[{'id': 1, 'product': 'Widget A', 'region': 'North', 'date': '2025-01-01', 'units_sold': 10, 'revenue': 100.0}, {'id': 2, 'product': 'Widget B', 'region': 'South', 'date': '2025-01-01', 'units_sold': 5, 'revenue': 50.0}]


In [10]:
!pip install fastapi uvicorn

  Using cached annotated_doc-0.0.4-py3-none-any.whl.metadata (6.6 kB)
Using cached annotated_doc-0.0.4-py3-none-any.whl (5.3 kB)

   ------------- -------------------------- 1/3 [starlette]
   ------------- -------------------------- 1/3 [starlette]
   ------------- -------------------------- 1/3 [starlette]
   ------------- -------------------------- 1/3 [starlette]
   ------------- -------------------------- 1/3 [starlette]
   ------------- -------------------------- 1/3 [starlette]
   ------------- -------------------------- 1/3 [starlette]
   ------------- -------------------------- 1/3 [starlette]
   ------------- -------------------------- 1/3 [starlette]
   -------------------------- ------------- 2/3 [fastapi]
   -------------------------- ------------- 2/3 [fastapi]
   -------------------------- ------------- 2/3 [fastapi]
   -------------------------- ------------- 2/3 [fastapi]
   -------------------------- ------------- 2/3 [fastapi]
   -------------------------- ----------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
